In [1]:
import os
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    StaleElementReferenceException,
)


def safe_click_elem(driver, elem, retries=6):
    last = None
    for _ in range(retries):
        try:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", elem)
            time.sleep(0.15)
            try:
                elem.click()
            except (ElementClickInterceptedException, StaleElementReferenceException):
                driver.execute_script("arguments[0].click();", elem)
            return
        except Exception as e:
            last = e
            time.sleep(0.4)
    raise last


def crawl_saramin_onepage_to_csv(keyword="데이터분석",
                                out_path="data_tmp/data_saramin.csv",
                                headless=False):

    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 20)

    try:
        driver.get("https://www.saramin.co.kr/")
        time.sleep(1)

        btn = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "button#btn_search.btn_search"))
        )
        safe_click_elem(driver, btn)

        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "input")))
        inputs = driver.find_elements(By.CSS_SELECTOR, "input")

        search_input = None
        for inp in inputs:
            t = (inp.get_attribute("type") or "").lower()
            if inp.is_displayed() and inp.is_enabled() and t in ["text", "search"]:
                search_input = inp
                break

        if search_input is None:
            raise RuntimeError("검색 입력창을 찾지 못했습니다.")

        safe_click_elem(driver, search_input)
        search_input.clear()
        search_input.send_keys(keyword)
        search_input.send_keys(Keys.ENTER)

        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.item_recruit")))

        items = driver.find_elements(By.CSS_SELECTOR, "div.item_recruit")
        rows = []

        for it in items:

            title, link = "", ""

            try:
                a = it.find_element(By.CSS_SELECTOR, "h2.job_tit > a")
                title = a.text.strip()
                link = a.get_attribute("href") or ""

                if link.startswith("/"):
                    link = "https://www.saramin.co.kr" + link

                link = link.replace("https://", "").replace("http://", "")
            except:
                pass

            company = ""

            try:
                comp = it.find_element(
                    By.CSS_SELECTOR,
                    "div.area_corp strong.corp_name > a"
                )
                company = comp.text.strip()
            except:
                pass

            cond_text = ""

            try:
                cond_parts = []
                spans = it.find_elements(By.CSS_SELECTOR, "div.job_condition > span")

                for idx, sp in enumerate(spans):

                    if idx == 0:
                        links = sp.find_elements(By.TAG_NAME, "a")

                        if links:
                            area = " ".join(
                                [x.text.strip() for x in links if x.text.strip()]
                            )
                            if area:
                                cond_parts.append(area)

                        else:
                            txt = sp.text.strip()
                            if txt:
                                cond_parts.append(txt)

                    else:
                        txt = sp.text.strip()
                        if txt:
                            cond_parts.append(txt)

                if cond_parts:
                    cond_text = "[" + ", ".join(cond_parts[:4]) + "]"

            except:
                cond_text = ""

            if title or company:
                rows.append({
                    "Site": "Saramin",
                    "Col_Company": company,
                    "Col_Recruit": title,
                    "Col_detail": cond_text,
                    "Col_url": link
                })

        df = pd.DataFrame(
            rows,
            columns=["Site", "Col_Company", "Col_Recruit", "Col_detail", "Col_url"]
        )

        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        df.to_csv(out_path, index=False, encoding="utf-8-sig")

        return df

    finally:
        driver.quit()


df_saramin = crawl_saramin_onepage_to_csv(
    keyword="데이터분석",
    out_path="data_tmp/data_saramin.csv",
    headless=False
)

display(df_saramin)

,Site,Col_Company,Col_Recruit,Col_detail,Col_url
0,Saramin,일루넥스,"(주)일루넥스 인공지능, 빅데이터 분석 경력 채용","[경기 고양시 덕양구, 경력 3~10년, 대졸↑, 정규직]",www.saramin.co.kr/zf_user/jobs/relay/view?view...
1,Saramin,(주)마크클라우드,"AI 개발(Python), 데이터분석 및 사업 기획 인턴 모집","[서울 강남구, 경력무관, 대졸↑, 인턴직]",www.saramin.co.kr/zf_user/jobs/relay/view?view...
2,Saramin,(주)엔틀,주)엔틀 에너지경영사업팀 데이터 분석 담당 채용(대구/서울),"[서울 성동구, 신입·경력, 초대졸↑, 정규직]",www.saramin.co.kr/zf_user/jobs/relay/view?view...
3,Saramin,나이스지니데이타주식회사,[NICE 지니데이타] 스마트팩토리/제조 분야 데이터 가공 분석,"[서울 영등포구, 경력2년↑, 대졸↑, 계약직]",www.saramin.co.kr/zf_user/jobs/relay/view?view...
4,Saramin,코리아크레딧뷰로(주),코리아크레딧뷰로(주) 데이터분석 및 운영 업무(계약직),"[서울 영등포구, 경력무관, 대졸↑, 기간제·계약직]",www.saramin.co.kr/zf_user/jobs/relay/view?view...
5,Saramin,코리아밸리(유),[정규직/본사근무] 광고 마케터 / 데이터분석 운영 담당자 모집,"[서울 중구, 경력무관, 고졸↑, 정규직]",www.saramin.co.kr/zf_user/jobs/relay/view?view...
6,Saramin,(주)스르르,[온라인MD] NUVN에서 성과/데이터 분석형 MD 채용,"[서울 마포구, 경력무관, 학력무관, 정규직]",www.saramin.co.kr/zf_user/jobs/relay/view?view...
7,Saramin,(주)이노션,[이노션] 데이터 분석 운영(GA4 Specialist) (계약직),"[서울 강남구, 경력1년↑, 대졸↑, 계약직]",www.saramin.co.kr/zf_user/jobs/relay/view?view...
8,Saramin,(주)에이치알비즈코리아,"데이터 분석( SQL, Python, R) MMORPG 모바일/세계적게임사","[경기 성남시 분당구, 경력 1~9년, 대졸↑, 정규직]",www.saramin.co.kr/zf_user/jobs/relay/view?view...
9,Saramin,주식회사 지니어스,[대기업계열사/마케팅에이전시] 디지털 데이터 분석가,"[서울 강남구, 경력 5~10년, 대졸↑, 정규직]",www.saramin.co.kr/zf_user/jobs/relay/view?view...
